# Certilab Adaptive RAG — Demo en vivo

Este notebook ejecuta el grafo **Adaptive RAG** canónico con 7 nodos y 2 loops
de auto-corrección, usando LangGraph + OpenAI + Tavily.

**Referencia**: [Building an Adaptive RAG System](https://levelup.gitconnected.com/building-an-adaptive-rag-system-with-langgraph-openai-and-tavily-c4ee39d2f021)

### Requisitos
- `OPENAI_API_KEY` (Colab: Secrets 🔑, local: `.env`)
- `TAVILY_API_KEY` opcional

💡 **Colab**: ejecutá la celda 1 para clonar e instalar. **Local**: salteala.


In [10]:
# === Solo para Colab: clonar repo e instalar dependencias ===
import os, sys
try:
    from google.colab import userdata
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    !git clone https://github.com/emersonheto/certilab-adaptive-rag.git /content/certilab-adaptive-rag
    %cd /content/certilab-adaptive-rag
    !pip install -q langgraph langchain-openai langchain-core openai tavily-python pydantic pydantic-settings python-dotenv
    print("✅ Repo clonado y dependencias instaladas")
else:
    print("ℹ️  Ejecutando en local — el repo ya está clonado")


fatal: destination path '/content/certilab-adaptive-rag' already exists and is not an empty directory.
/content/certilab-adaptive-rag
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00
✅ Repo clonado y dependencias instaladas


In [11]:
# === Configurar API keys ===
if _IN_COLAB:
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY (sk-...): ").strip()
    try:
        os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    except Exception:
        os.environ["TAVILY_API_KEY"] = input("🔑 TAVILY_API_KEY (tvly-...) o Enter para saltar: ").strip() or None
    print("✅ Keys configuradas")
else:
    from dotenv import load_dotenv
    from pathlib import Path
    _env = Path.cwd() / ".env"
    load_dotenv(_env if _env.exists() else None, override=False)

if not os.getenv("OPENAI_API_KEY"):
    print("❌ OPENAI_API_KEY no encontrada")
    sys.exit(1)
print(f"✅ OPENAI_API_KEY | TAVILY: {'✅' if os.getenv('TAVILY_API_KEY') else '❌ (web search desactivado)'}")


✅ Keys configuradas
✅ OPENAI_API_KEY configurada | TAVILY: ❌


In [12]:
from app.adaptive_rag.graph import build_graph
from app.adaptive_rag.state import AdaptiveRAGState
from app.config import Settings
from app.domain.models import Role
from app.security.access_control import Principal, scope_from_principal
from app.tools.web_search import TavilyWebSearch, WebSearchConfig
print("✅ Imports OK")


✅ Imports OK


In [13]:
from pathlib import Path
settings = Settings()
print(f"Modo: {settings.app_mode}")

if settings.app_mode != "real":
    # Mock mode — datos en memoria
    from app.ingestion.loader import load_certificates, load_customers, load_histories, load_pdf_texts
    from app.ingestion.splitter import build_pdf_chunks
    from app.ingestion.indexer import InMemoryVectorIndex
    data_dir = Path("data")
    certificates = load_certificates(data_dir)
    load_customers(data_dir); load_histories(data_dir)
    chunks = build_pdf_chunks(certificates, load_pdf_texts(data_dir))
    index = InMemoryVectorIndex(chunks=chunks)
    embedding_provider = None
    print(f"✅ Mock: {len(certificates)} certificados en memoria")
else:
    from app.retrieval.qdrant_index import QdrantVectorIndex
    from app.tools.embeddings import EmbeddingProviderConfig, EmbeddingsProvider
    embedding_provider = EmbeddingsProvider(EmbeddingProviderConfig.from_settings(settings))
    index = QdrantVectorIndex.from_settings(settings, embedding_provider)
    print(f"✅ Qdrant conectado | {settings.openai_embedding_model}")

tavily_key = settings.tavily_api_key
web_search = TavilyWebSearch(WebSearchConfig(tavily_api_key=tavily_key))
graph = build_graph(index=index, embeddings=embedding_provider, web_search=web_search, settings=settings)
print(f"✅ Grafo compilado | Web search: {'✅' if tavily_key else '❌'}")


Modo: mock
✅ Mock: 3 certificados en memoria
✅ Grafo compilado | Web search: ❌


In [14]:
def make_state(question: str) -> AdaptiveRAGState:
    p = Principal(role=Role.ADMIN, customer_id=None, user_id=1)
    s = scope_from_principal(p)
    return {"question": question, "generation": "", "documents": [],
            "web_results": [], "route": "", "rewrite_count": 0,
            "regenerate_count": 0, "hallucination_verdict": "",
            "answer_verdict": "", "principal": p, "scope": s}

def run(question: str):
    for step in graph.stream(make_state(question)):
        for name, out in step.items():
            print(f"--- {name} ---")
            if isinstance(out, dict):
                for k, v in out.items():
                    s = repr(v); print(f"  {k}: {s[:180]}")
            print()
print("✅ Helpers")


✅ Helpers


## 🔍 Query A: Vectorstore — Procedimiento INDECOPI


In [ ]:
run("¿Qué procedimiento de calibración sigue la norma INDECOPI?")


--- route_question ---
  route: 'vectorstore'

--- retrieve ---
  documents: ['Certificado de calibración demo CERT-2025-002. Equipo: termómetro digital para monitoreo de temperatura. El procedimiento comparó lecturas en puntos de referencia controlados y r

--- grade_documents ---
  documents: []

--- transform_query ---
  question: '¿Cuál es el procedimiento de calibración que sigue la norma INDECOPI en el laboratorio?'
  rewrite_count: 1

--- retrieve ---
  documents: ['ara el proceso del cliente demo. Observación técnica: se recomienda conservar el equipo en superficie nivelada, evitar corrientes de aire y repetir la calibración según el interv

--- grade_documents ---
  documents: []

--- transform_query ---
  question: '¿Podrían detallar el procedimiento de calibración que sigue la norma INDECOPI en el laboratorio de calibración?'
  rewrite_count: 2

--- retrieve ---
  documents: ['ara el proceso del cliente demo. Observación técnica: se recomienda conservar el equipo en superfic

## 🎯 Query B: Tenant isolation — ALS PERU


In [ ]:
run("¿Qué certificados tiene ALS PERU?")


## 🔄 Self-Correction Loops
1. **Rewrite loop**: docs irrelevantes → reescribe → reintenta (máx 3)
2. **Regenerate loop**: alucina → regenera (máx 2). No útil → reescribe


### 🔄 Loop 1: Reescritura


In [ ]:
run("Dame info de calibración")


### 📊 Query D: Datos de tablas — Medición 105°C


In [ ]:
run("¿Cuál fue la temperatura máxima registrada a 105°C?")


### 🏷️ Query E: Metadatos — Fecha + Tipo


In [ ]:
run("¿Qué certificados acreditados se emitieron en mayo 2026?")


## 📊 Visualización del grafo


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## ✅ Resumen
- ✅ Enrutamiento adaptativo
- ✅ Tenant isolation
- ✅ Gradeo de relevancia
- ✅ Reescritura de query (máx 3)
- ✅ Verificación de alucinaciones (máx 2)
- ✅ Datos reales: 154 certificados
- ✅ Pipeline: PyMuPDF + Camelot + Unstructured → Qdrant

**Stack**: Python 3.11 | LangGraph | OpenAI | Tavily | Pydantic v2 | Qdrant
